# Finance Guidance Assistant — RAG Prototype (Colab)

Run all cells top to bottom. Make sure a GPU runtime is selected:
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

This notebook:
1. Installs dependencies
2. Gets the project code (clone from GitHub, or upload manually)
3. Builds the FAISS index from the finance knowledge base
4. Runs a quick retrieval sanity check
5. Launches the Gradio app (shareable public link)
6. Runs the evaluation suite and shows results


# 1. Dependencies

In [4]:
!pip install sentence-transformers faiss-cpu transformers torch gradio accelerate --quiet

# 2. Clone from GitHub

In [5]:
GITHUB_REPO_URL = "https://github.com/SheerinMM/finance-guidance-rag.git"

!git clone {GITHUB_REPO_URL}
%cd finance-rag

fatal: destination path 'finance-guidance-rag' already exists and is not an empty directory.
[Errno 2] No such file or directory: 'finance-rag'
/content


## 3. Build the FAISS index

In [8]:
!pwd
!ls

/content
finance-guidance-rag  sample_data


In [9]:
%cd finance-guidance-rag

/content/finance-guidance-rag


In [13]:
!ls -la
!ls -la src

total 44
drwxr-xr-x 7 root root 4096 Jul  4 02:14 .
drwxr-xr-x 1 root root 4096 Jul  4 02:14 ..
drwxr-xr-x 2 root root 4096 Jul  4 02:14 data
drwxr-xr-x 2 root root 4096 Jul  4 02:14 eval
drwxr-xr-x 8 root root 4096 Jul  4 02:14 .git
-rw-r--r-- 1 root root  224 Jul  4 02:14 .gitignore
drwxr-xr-x 2 root root 4096 Jul  4 02:14 notebooks
-rw-r--r-- 1 root root 5985 Jul  4 02:14 README.md
-rw-r--r-- 1 root root  114 Jul  4 02:14 requirements.txt
drwxr-xr-x 2 root root 4096 Jul  4 02:14 src
total 32
drwxr-xr-x 2 root root 4096 Jul  4 02:14 .
drwxr-xr-x 7 root root 4096 Jul  4 02:14 ..
-rw-r--r-- 1 root root 2422 Jul  4 02:14 app.py
-rw-r--r-- 1 root root 2275 Jul  4 02:14 chunking.py
-rw-r--r-- 1 root root 2166 Jul  4 02:14 embed_index.py
-rw-r--r-- 1 root root 3418 Jul  4 02:14 generate.py
-rw-r--r-- 1 root root 1475 Jul  4 02:14 retrieve.py
-rw-r--r-- 1 root root 3667 Jul  4 02:14 safeguard.py


In [14]:
import os, sys

os.chdir('/content/finance-guidance-rag')  # force absolute path, no ambiguity
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # use absolute path too

print("cwd:", os.getcwd())
print("src exists:", os.path.exists('src/chunking.py'))
print("sys.path[0:2]:", sys.path[0:2])

import chunking
print("Import worked:", chunking.__file__)

cwd: /content/finance-guidance-rag
src exists: True
sys.path[0:2]: ['/content/finance-guidance-rag/src', 'src']
Import worked: /content/finance-guidance-rag/src/chunking.py


In [15]:
import os, sys

os.chdir('/content/finance-guidance-rag')
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from chunking import build_chunk_corpus
from embed_index import build_index, save_index

corpus = build_chunk_corpus('data/finance_kb.json')
print(f"Loaded {len(corpus)} chunks from knowledge base.")

index, model, corpus = build_index(corpus)
save_index(index, corpus, 'data/faiss_index.bin', 'data/corpus.json')
print(f"Built FAISS index with {index.ntotal} vectors (dim={index.d}).")

Loaded 27 chunks from knowledge base.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Built FAISS index with 27 vectors (dim=384).


## 4. Quick retrieval sanity check

In [ ]:
from retrieve import retrieve

test_queries = [
    "When can I take money out of my pension?",
    "What's the difference between saving and investing?",
    "How do I improve my credit score?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, index, model, corpus, top_k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['title']} ({r['category']})")

## 5. Test the safeguard

In [ ]:
from safeguard import check_safeguards

test_cases = [
    "Should I transfer my pension into a SIPP given my situation?",
    "What is a Lifetime ISA?",
]

for q in test_cases:
    retrieved = retrieve(q, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(q, retrieved)
    print(f"Query: {q}")
    print(f"  Blocked: {blocked}")
    if blocked:
        print(f"  Message: {message}")
    print()

## 6. Load the generation model and test end-to-end

In [ ]:
from generate import Generator, format_answer_with_citations

generator = Generator()  # downloads Qwen2.5-1.5B-Instruct on first run

query = "What is a Lifetime ISA?"
retrieved = retrieve(query, index, model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

## 7. Launch the Gradio app

In [ ]:
import gradio as gr

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(
    fn=answer_question,
    title="Finance Guidance Assistant",
    description="Ask general questions about UK pensions, savings, debt, mortgages, credit, and investing. This is guidance, not regulated financial advice.",
    examples=[
        "What is the difference between a defined contribution and defined benefit pension?",
        "What is a Lifetime ISA?",
        "How can I recognise a pension scam?",
        "Should I move my pension into a SIPP given my situation?",
    ],
)

demo.launch(share=True)  # take a screenshot of the public link + a sample conversation for your report

## 8. Run the evaluation suite

Stop the Gradio cell above first (interrupt it), then run this.


In [ ]:
%cd ..
!python eval/evaluate.py
%cd finance-rag

## 9. View evaluation results

These numbers go straight into your Part B technical report's evaluation section.


In [ ]:
import pandas as pd

retrieval_df = pd.read_csv('eval/retrieval_results.csv')
print("Retrieval accuracy:", retrieval_df['retrieval_hit'].mean())
print("Average retrieval latency (s):", retrieval_df['retrieval_latency_sec'].mean())
retrieval_df

In [ ]:
generation_df = pd.read_csv('eval/generation_results.csv')
generation_df
# Fill in the 'manual_grade' column (correct / partial / incorrect) by reading each
# generated_answer against the ground_truth, then re-load this CSV to compute
# an overall accuracy percentage for your report.

In [ ]:
safeguard_df = pd.read_csv('eval/safeguard_results.csv')
safeguard_df